# ClusterAPI

Cluster API ist ein Kubernetes-Projekt zur deklarativen Erstellung und Verwaltung von Kubernetes-Clustern. Es bildet den gesamten Cluster-Lifecycle über Kubernetes-Ressourcen und Controller ab.

Die Ressourcen sind hierarchisch aufgebaut: Ein `Cluster` beschreibt den logischen Kubernetes-Cluster. Die Control Plane wird über eine Ressource wie `KubeadmControlPlane` verwaltet, während Worker Nodes typischerweise über `MachineDeployments`, darunterliegende `MachineSets` und einzelne `Machines` abgebildet werden. Jede `Machine` repräsentiert dabei eine konkrete virtuelle oder physische Instanz, die später als Kubernetes Node registriert wird.

```text
Cluster
├── ControlPlane
│   └── Machines
└── MachineDeployments
    └── MachineSets
        └── Machines
            └── Kubernetes Nodes
```

Im Gegensatz zu Terraform prüft Cluster API laufend, ob der tatsächliche Zustand dem gewünschten Zustand entspricht, und korrigiert Abweichungen automatisch. Es ermöglicht einheitliche Cluster-Vorlagen, kontrollierte Updates, das Skalieren von Nodes und den automatischen Ersatz defekter Maschinen.

Da Cluster, Control Plane und Worker Nodes als Kubernetes-Ressourcen verwaltet werden, lassen sie sich direkt mit GitOps, Berechtigungen, Richtlinien und Monitoring verbinden. 

**Der grösste Vorteil liegt deshalb nicht nur in der Erstellung von Clustern, sondern vor allem in deren einheitlichem und automatisiertem Betrieb.**

---

Installation CLI

In [ ]:
%%bash
curl -L https://github.com/kubernetes-sigs/cluster-api/releases/download/v1.13.3/clusterctl-linux-amd64 -o clusterctl
sudo install -o root -g root -m 0755 clusterctl /usr/local/bin/clusterctl

### Load Balancer

Im Zusammenhang mit Cluster API wird ein Load Balancer benötigt, damit die automatisch erstellten Control-Plane-Nodes über eine gemeinsame, feste Adresse erreichbar sind. 

Cluster API kann dadurch Control-Plane-Nodes austauschen oder erweitern, ohne dass sich der Zugriffspunkt auf den Kubernetes-API-Server ändert.

In [ ]:
%%bash
METALLB_VER=$(curl "https://api.github.com/repos/metallb/metallb/releases/latest" | jq -r ".tag_name")
kubectl apply -f "https://raw.githubusercontent.com/metallb/metallb/${METALLB_VER}/config/manifests/metallb-native.yaml"
kubectl wait pods -n metallb-system -l app=metallb,component=controller --for=condition=Ready --timeout=10m
kubectl wait pods -n metallb-system -l app=metallb,component=speaker --for=condition=Ready --timeout=2m
cat <<EOF | kubectl apply -f -
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: capi-ip-pool
  namespace: metallb-system
spec:
  addresses:
  - 10.10.0.30-10.10.0.90
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: empty
  namespace: metallb-system
EOF


### Cluster Initialisieren

Initialisieren des Management-Cluster mit dem KubeVirt-Provider. 

Dabei werden die benötigten Cluster-API-Komponenten sowie die Controller und Custom Resources für KubeVirt installiert. Der Management-Cluster kann anschliessend KubeVirt-basierte Workload-Cluster erstellen, verwalten, skalieren und aktualisieren.


In [ ]:
%%bash
clusterctl init --infrastructure kubevirt

export NODE_VM_IMAGE_TEMPLATE="quay.io/capk/ubuntu-2404-container-disk:v1.32.1"
export CAPK_GUEST_K8S_VERSION="${NODE_VM_IMAGE_TEMPLATE/*:/}"
export CRI_PATH="unix:///var/run/containerd/containerd.sock"
    
clusterctl generate cluster capi-quickstart \
  --infrastructure="kubevirt" \
  --flavor lb \
  --kubernetes-version ${CAPK_GUEST_K8S_VERSION} \
  --control-plane-machine-count=1 \
  --worker-machine-count=2 \
  > capi-quickstart.yaml


In [ ]:
%%bash
kubectl apply -f capi-quickstart.yaml

### Workaround für den fehlenden Containerd-Pfad bei den Worker Nodes

Im generierten Worker-Template ist der CRI-Socket für Containerd nicht explizit definiert. Dadurch kann kubeadm beim Beitritt eines Worker Nodes unter Umständen nicht erkennen, welche Container-Runtime verwendet werden soll.

Mit der folgenden Anpassung wird der Containerd-Socket im KubeadmConfigTemplate fest eingetragen. Neue Worker Nodes verwenden damit beim Cluster-Beitritt den korrekten CRI-Endpunkt.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: bootstrap.cluster.x-k8s.io/v1beta1
kind: KubeadmConfigTemplate
metadata:
  name: capi-quickstart-md-0
  namespace: default
spec:
  template:
    spec:
      joinConfiguration:
        nodeRegistration:
          criSocket: unix:///var/run/containerd/containerd.sock
          kubeletExtraArgs: {}
EOF

### Status der erstellten Ressourcen prüfen

Nach dem Erstellen des Workload-Clusters können die wichtigsten Cluster-API- und KubeVirt-Ressourcen angezeigt werden:

Dabei ist insbesondere der Status der `KubeadmControlPlane` relevant. Die Control Plane muss vollständig bereit sein, bevor auf den neuen Workload-Cluster zugegriffen werden kann. Der Aufbau kann einige Zeit dauern, da Cluster API zuerst die virtuellen Maschinen erstellt, Kubernetes initialisiert und die Control-Plane-Nodes registriert.


In [ ]:
%%bash
kubectl get vm,vmi,cluster,kubeadmcontrolplane

Sobald die `KubeadmControlPlane` bereit ist, kann die Kubeconfig des Workload-Clusters erzeugt werden:

Anschliessend kann mit dieser Kubeconfig auf den neu erstellten Cluster zugegriffen und geprüft werden, ob alle Systemkomponenten erfolgreich gestartet wurden:

In [ ]:
%%bash
clusterctl get kubeconfig capi-quickstart > capi-quickstart.kubeconfig
kubectl --kubeconfig capi-quickstart.kubeconfig get pods -A


### Calico als Overlay-Netzwerk installieren

Damit Pods auf unterschiedlichen Worker Nodes miteinander kommunizieren können, benötigt der Workload-Cluster ein Container Network Interface, kurz CNI. In diesem Beispiel wird Calico verwendet.

Da die Kubernetes-Nodes als virtuelle Maschinen innerhalb von KubeVirt betrieben werden, wird für den Pod-Verkehr ein VXLAN-Overlay-Netzwerk eingesetzt. Dabei wird der Netzwerkverkehr zwischen den Nodes gekapselt und über das bestehende Netzwerk des KubeVirt-Clusters transportiert. Das darunterliegende Netzwerk muss dadurch die Pod-Netzwerke nicht direkt kennen oder routen können.

Die Anpassungen haben folgende Aufgaben:

* CALICO_IPV4POOL_CIDR definiert mit 10.243.0.0/16 den IP-Adressbereich für die Pods.
* CLUSTER_TYPE wird auf k8s gesetzt, da Calico in einem normalen Kubernetes-Cluster eingesetzt wird.
* IP-in-IP wird deaktiviert, da für das Overlay-Netzwerk VXLAN verwendet wird.
* VXLAN wird aktiviert, damit der Pod-Verkehr zwischen den virtuellen Nodes gekapselt übertragen werden kann.
* Der VXLAN-Port wird auf 6789 gesetzt, damit er zur vorgesehenen Netzwerkkonfiguration der KubeVirt-Umgebung passt.

In [ ]:
%%bash
curl https://raw.githubusercontent.com/projectcalico/calico/v3.29.1/manifests/calico.yaml -o calico-workload.yaml

sed -i -E 's|^( +)# (- name: CALICO_IPV4POOL_CIDR)$|\1\2|g;'\
's|^( +)# (  value: )"192.168.0.0/16"|\1\2"10.243.0.0/16"|g;'\
'/- name: CLUSTER_TYPE/{ n; s/( +value: ").+/\1k8s"/g };'\
'/- name: CALICO_IPV4POOL_IPIP/{ n; s/value: "Always"/value: "Never"/ };'\
'/- name: CALICO_IPV4POOL_VXLAN/{ n; s/value: "Never"/value: "Always"/};'\
'/# Set Felix endpoint to host default action to ACCEPT./a\            - name: FELIX_VXLANPORT\n              value: "6789"' \
calico-workload.yaml
kubectl --kubeconfig=./capi-quickstart.kubeconfig create -f calico-workload.yaml


In [ ]:
%%bash
export KUBECONFIG=`pwd`/capi-quickstart.kubeconfig
kubectl get nodes
kubectl get pods -A -o wide

### Headlamp starten

Anschliessend wird Headlamp gestartet, um den Workload-Cluster über eine grafische Oberfläche zu überwachen und Kubernetes-Ressourcen einfacher zu verwalten.

Die IP-Adresse des Clusters ist ggf. anzupassen.

In [ ]:
%%bash
export KUBECONFIG=`pwd`/capi-quickstart.kubeconfig
kubectl apply -f https://raw.githubusercontent.com/headlamp-k8s/headlamp/main/kubernetes-headlamp.yaml
kubectl -n kube-system create serviceaccount headlamp-admin
kubectl create clusterrolebinding headlamp-admin --serviceaccount=kube-system:headlamp-admin --clusterrole=cluster-admin

kubectl patch svc headlamp \
  -n kube-system \
  --type='merge' \
  -p '{
    "spec": {
      "type": "NodePort",
      "ports": [
        {
          "port": 80,
          "targetPort": 4466,
          "nodePort": 30444
        }
      ]
    }
  }'
echo "HeadLamp: http://"10.10.0.31":30444"
kubectl create token headlamp-admin -n kube-system --duration=48h

---

### Aufräumen


In [ ]:
%%bash
kubectl delete cluster capi-quickstart